In [2]:
!pip install rank_bm25 nltk pymorphy3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 60.2 MB/s eta 0:00:00


In [3]:
import re
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from rank_bm25 import BM25Okapi
from pymorphy3 import MorphAnalyzer
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")

morph = MorphAnalyzer()
stop_words = set(stopwords.words("russian"))

articles = pd.read_feather("articles.f")
calibration = pd.read_feather("calibration.f")
test = pd.read_feather("test.f")


def clean_html(text):
    if pd.isna(text):
        return ""
    return BeautifulSoup(text, "html.parser").get_text(separator=" ", strip=True)


articles["clean_body"] = articles["body"].apply(clean_html)

articles["search_text"] = (
    articles["title"].fillna("") + " " +
    articles["title"].fillna("") + " " +
    articles["title"].fillna("") + " " +
    articles["clean_body"]
)

articles["dense_text"] = articles["title"].fillna("") + ". " + articles["clean_body"]


def tokenize(text):
    text = str(text).lower()
    text = text.replace("ё", "е")
    text = re.sub(r"[^а-яa-z0-9 ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = []
    for word in text.split():
        if len(word) < 2:
            continue
        lemma = morph.normal_forms(word)[0]
        if lemma in stop_words:
            continue
        tokens.append(lemma)
    return tokens


tokenized_corpus = articles["search_text"].apply(tokenize).tolist()
bm25 = BM25Okapi(tokenized_corpus, k1=1.8, b=0.8)
article_ids = articles["article_id"].to_numpy()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [4]:
def average_precision_at_k(y_true, y_pred, k=10):
    y_true_set = set(y_true)
    y_pred = y_pred[:k]
    if not y_true_set:
        return 0.0
    hits = 0
    score = 0.0
    for i, doc_id in enumerate(y_pred):
        if doc_id in y_true_set:
            hits += 1
            score += hits / (i + 1)
    return score / min(len(y_true_set), k)


def mean_average_precision_at_k(y_true_list, y_pred_list, k=10):
    return float(np.mean([
        average_precision_at_k(t, p, k) for t, p in zip(y_true_list, y_pred_list)
    ]))


def top_k_ids(scores, k=10):
    top_idx = np.argsort(scores)[::-1][:k]
    return article_ids[top_idx].tolist()


gt = calibration["ground_truth"].apply(lambda x: [int(i) for i in x.split()]).tolist()

bm25_only_preds = [
    top_k_ids(bm25.get_scores(tokenize(q))) for q in calibration["query_text"]
]
print("BM25-only MAP@10 (calibration):", mean_average_precision_at_k(gt, bm25_only_preds))

BM25-only MAP@10 (calibration): 0.2863224206349206


In [5]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("intfloat/multilingual-e5-base")

articles["dense_text"] = "passage: " + articles["title"].fillna("") + ". " + articles["clean_body"]

article_embeddings = embed_model.encode(
    articles["dense_text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

def minmax(x):
    x = np.asarray(x, dtype=float)
    lo, hi = x.min(), x.max()
    if hi - lo < 1e-12:
        return np.zeros_like(x)
    return (x - lo) / (hi - lo)

def compute_scores(query_texts):
    query_texts = list(query_texts)
    dense_q = embed_model.encode(
        ["query: " + q for q in query_texts],
        batch_size=64, show_progress_bar=True, normalize_embeddings=True,
    )
    dense_scores = dense_q @ article_embeddings.T
    bm25_scores = np.vstack([bm25.get_scores(tokenize(q)) for q in query_texts])
    return bm25_scores, dense_scores


def combine_and_rank(bm25_scores, dense_scores, alpha, k=10):
    preds = []
    for i in range(bm25_scores.shape[0]):
        combined = alpha * minmax(bm25_scores[i]) + (1 - alpha) * minmax(dense_scores[i])
        preds.append(top_k_ids(combined, k))
    return preds


calib_bm25_scores, calib_dense_scores = compute_scores(calibration["query_text"])

best_alpha, best_score = 0.0, -1.0
for alpha in np.arange(0.0, 1.01, 0.05):
    preds = combine_and_rank(calib_bm25_scores, calib_dense_scores, alpha)
    score = mean_average_precision_at_k(gt, preds)
    if score > best_score:
        best_alpha, best_score = alpha, score

print(f"лучший alpha = {best_alpha:.2f}, MAP@10 (calibration) = {best_score:.4f}")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

...

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

лучший alpha = 0.50, MAP@10 (calibration) = 0.3225


In [6]:
test_bm25_scores, test_dense_scores = compute_scores(test["query_text"])
test_preds = combine_and_rank(test_bm25_scores, test_dense_scores, best_alpha)

source_data = test.copy()
source_data["answer"] = [" ".join(map(str, p)) for p in test_preds]
source_data[["query_id", "answer"]].to_csv("answer.csv", index=False)

Batches:   0%|          | 0/8 [00:00<?, ?it/s]